In [0]:
-- ============================================================
-- Project: Databricks SQL Retail Analytics Dashboard
-- File: 02_dashboard_queries.sql
-- Purpose: Queries used to create dashboard visualizations
-- ============================================================

USE SCHEMA retail_analytics_project;


-- ============================================================
-- 1. Monthly sales trend
-- ============================================================

SELECT
    d.year,
    d.month,
    d.month_name,
    d.year_month,

    COUNT(DISTINCT f.invoice_no) AS total_invoices,

    SUM(f.sold_quantity) AS sold_units,

    SUM(f.cancelled_quantity) AS cancelled_units,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(f.cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_date d
    ON f.invoice_date = d.date

GROUP BY
    d.year,
    d.month,
    d.month_name,
    d.year_month

ORDER BY
    d.year_month;


-- ============================================================
-- 2. Sales performance by day of week
-- ============================================================

SELECT
    d.day_of_week_number,
    d.day_name,

    COUNT(DISTINCT f.invoice_no) AS total_invoices,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_date d
    ON f.invoice_date = d.date

GROUP BY
    d.day_of_week_number,
    d.day_name

ORDER BY
    d.day_of_week_number;


-- ============================================================
-- 3. Weekday vs weekend performance
-- ============================================================

SELECT
    CASE
        WHEN d.is_weekend
        THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,

    COUNT(DISTINCT f.invoice_no) AS total_invoices,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_date d
    ON f.invoice_date = d.date

GROUP BY
    CASE
        WHEN d.is_weekend
        THEN 'Weekend'
        ELSE 'Weekday'
    END

ORDER BY
    net_sales DESC;


-- ============================================================
-- 4. Sales by country
-- ============================================================

SELECT
    country,

    COUNT(DISTINCT invoice_no) AS total_invoices,

    COUNT(DISTINCT CASE
        WHEN customer_id <> 'UNKNOWN'
        THEN customer_id
    END) AS known_customers,

    SUM(sold_quantity) AS sold_units,

    SUM(cancelled_quantity) AS cancelled_units,

    SUM(net_quantity) AS net_units,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales

GROUP BY
    country

ORDER BY
    net_sales DESC;


-- ============================================================
-- 5. United Kingdom vs international market
-- ============================================================

SELECT
    CASE
        WHEN country = 'United Kingdom'
        THEN 'United Kingdom'
        ELSE 'International'
    END AS market_group,

    COUNT(DISTINCT invoice_no) AS total_invoices,

    COUNT(DISTINCT CASE
        WHEN customer_id <> 'UNKNOWN'
        THEN customer_id
    END) AS known_customers,

    SUM(sold_quantity) AS sold_units,

    SUM(cancelled_quantity) AS cancelled_units,

    SUM(net_quantity) AS net_units,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales

GROUP BY
    CASE
        WHEN country = 'United Kingdom'
        THEN 'United Kingdom'
        ELSE 'International'
    END

ORDER BY
    net_sales DESC;


-- ============================================================
-- 6. Top international countries by net sales
-- United Kingdom excluded
-- ============================================================

SELECT
    country,

    COUNT(DISTINCT invoice_no) AS total_invoices,

    COUNT(DISTINCT CASE
        WHEN customer_id <> 'UNKNOWN'
        THEN customer_id
    END) AS known_customers,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales

WHERE country <> 'United Kingdom'

GROUP BY
    country

ORDER BY
    net_sales DESC

LIMIT 10;


-- ============================================================
-- 7. Top products by net sales
-- Requires DOT and other service codes to be marked as
-- is_non_product_code = true in dim_product
-- ============================================================

SELECT
    f.stock_code,

    p.canonical_product_description,

    COUNT(DISTINCT f.invoice_no) AS total_invoices,

    SUM(f.sold_quantity) AS sold_units,

    SUM(f.cancelled_quantity) AS cancelled_units,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(f.cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_product p
    ON f.stock_code = p.stock_code

WHERE p.is_non_product_code = false

GROUP BY
    f.stock_code,
    p.canonical_product_description

ORDER BY
    net_sales DESC

LIMIT 10;


-- ============================================================
-- 8. Top products by net units
-- ============================================================

SELECT
    f.stock_code,

    p.canonical_product_description,

    SUM(f.sold_quantity) AS sold_units,

    SUM(f.cancelled_quantity) AS cancelled_units,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_product p
    ON f.stock_code = p.stock_code

WHERE p.is_non_product_code = false

GROUP BY
    f.stock_code,
    p.canonical_product_description

ORDER BY
    net_units DESC

LIMIT 10;


-- ============================================================
-- 9. Top customers by net sales
-- ============================================================

SELECT
    f.customer_id,

    c.primary_country,

    COUNT(DISTINCT f.invoice_no) AS total_invoices,

    SUM(f.sold_quantity) AS sold_units,

    SUM(f.cancelled_quantity) AS cancelled_units,

    SUM(f.net_quantity) AS net_units,

    ROUND(
        SUM(f.gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(f.cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales f

INNER JOIN dim_customer c
    ON f.customer_id = c.customer_id

WHERE c.is_known_customer = true

GROUP BY
    f.customer_id,
    c.primary_country

ORDER BY
    net_sales DESC

LIMIT 10;


-- ============================================================
-- 10. Known vs unknown customer revenue
-- ============================================================

SELECT
    CASE
        WHEN is_customer_known
        THEN 'Known Customer'
        ELSE 'Unknown Customer'
    END AS customer_type,

    COUNT(*) AS total_lines,

    COUNT(DISTINCT invoice_no) AS total_invoices,

    SUM(net_quantity) AS net_units,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales,

    ROUND(
        SUM(net_sales_amount)
        / NULLIF(
            SUM(SUM(net_sales_amount)) OVER (),
            0
        )
        * 100.0,
        2
    ) AS net_sales_share_pct

FROM fact_sales

GROUP BY
    CASE
        WHEN is_customer_known
        THEN 'Known Customer'
        ELSE 'Unknown Customer'
    END

ORDER BY
    net_sales DESC;


-- ============================================================
-- 11. Largest commercial transactions
-- ============================================================

SELECT
    invoice_no,
    transaction_type,
    customer_id,
    country,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS invoice_net_sales,

    SUM(net_quantity) AS invoice_net_units

FROM fact_sales

GROUP BY
    invoice_no,
    transaction_type,
    customer_id,
    country

ORDER BY
    ABS(SUM(net_sales_amount)) DESC

LIMIT 10;


-- ============================================================
-- 12. Cancellation rate by country
-- Only markets with at least 10,000 in gross sales
-- ============================================================

SELECT
    country,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales,

    ROUND(
        SUM(cancellation_amount)
        / NULLIF(SUM(gross_sales_amount), 0)
        * 100.0,
        2
    ) AS cancellation_value_rate_pct

FROM fact_sales

GROUP BY
    country

HAVING
    SUM(gross_sales_amount) >= 10000

ORDER BY
    cancellation_value_rate_pct DESC;


-- ============================================================
-- 13. Customers with highest cancellation amount
-- ============================================================

SELECT
    f.customer_id,

    c.primary_country,

    COUNT(DISTINCT CASE
        WHEN f.transaction_type = 'CANCELLATION'
        THEN f.invoice_no
    END) AS cancellation_invoices,

    SUM(f.cancelled_quantity) AS cancelled_units,

    ROUND(
        SUM(f.gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(f.cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(f.net_sales_amount),
        2
    ) AS net_sales,

    ROUND(
        SUM(f.cancellation_amount)
        / NULLIF(SUM(f.gross_sales_amount), 0)
        * 100.0,
        2
    ) AS cancellation_value_rate_pct

FROM fact_sales f

INNER JOIN dim_customer c
    ON f.customer_id = c.customer_id

WHERE c.is_known_customer = true

GROUP BY
    f.customer_id,
    c.primary_country

HAVING
    SUM(f.cancellation_amount) > 0

ORDER BY
    cancellation_amount DESC

LIMIT 20;


-- ============================================================
-- 14. Gold data quality overview
-- ============================================================

SELECT
    total_silver_rows,
    unknown_customer_rows,
    unknown_product_rows,
    sale_rows,
    cancellation_rows,
    inventory_adjustment_rows,
    non_commercial_rows,
    bad_debt_adjustment_amount,
    gold_processed_ts

FROM gold_data_quality_summary;